# Bloom Craft — เครื่องเจนรูปแยก (SDXL)

รันบน **Colab อีกบัญชี** (ไม่ใช่บัญชีที่รัน notebook วิเคราะห์) · Runtime → Change runtime type → **T4 GPU** → Run all

1. รอเซลล์สุดท้ายขึ้นลิงก์ `https://....trycloudflare.com`
2. คัดลอกลิงก์ไปใส่ `GEN_URL = "..."` ในข้อ 2) ตั้งค่า ของ notebook วิเคราะห์ (`flower_counter_demo_4ชนิด_v6_แบ่งโซน.ipynb`) แล้วค่อย Run all ฝั่งนั้น
3. เปิดเว็บจากลิงก์ของ notebook วิเคราะห์ตามปกติ — กด Generate แล้วงานเจนจะวิ่งมาที่นี่

ทำไมแยก: SDXL ใหญ่ ~7GB ถ้าอยู่เครื่องเดียวกับ SAM 3 แรม Colab เต็มจนแครช · เครื่องนี้ไม่ต้องใช้ HF token


In [ ]:
%pip install -q diffusers accelerate fastapi uvicorn

## 1) โหลด SDXL + IP-Adapter

In [ ]:
import base64, io, re, os, subprocess, threading, time
import torch
from PIL import Image, ImageDraw
from diffusers import AutoencoderKL, DPMSolverMultistepScheduler, StableDiffusionXLPipeline, StableDiffusionXLInpaintPipeline
from diffusers.image_processor import IPAdapterMaskProcessor
from transformers import CLIPVisionModelWithProjection

GEN_MODEL = "SG161222/RealVisXL_V5.0"   # SDXL สายภาพถ่ายจริง
GEN_STEPS = 25
GEN_W, GEN_H = 832, 1024   # แนวตั้ง (ขนาดที่ SDXL ถนัด)
REF_SCALE = 0.30            # ดอกชนิดเดียว
REF_SCALE_ZONE = 0.26      # หลายชนิด: ปรับลดเพื่อไม่ให้ดึงลักษณะดอกเดี่ยวยักษ์มาบวมทับ และให้โมเดลจัดช่อผสมได้กลมกลืน
REFINE_STRENGTH = 0.38     # รอบวาดซ้ำทีละคลัสเตอร์: คมชัดขึ้น ดอกไม้แซมกันกลมกลืน ไม่ทำลายโครงสร้างช่อรวมและแจกัน
# shortcut: เว็บผ่าน Cloudflare ตัดที่ 100 วินาที → 4 ชนิด (4 โซน) บน T4 อาจเกิน; ถ้าเจอ error 524/502 ลด GEN_STEPS

assert torch.cuda.is_available(), "ต้องเปิด GPU: Runtime → Change runtime type → T4 GPU"
pipe = StableDiffusionXLPipeline.from_pretrained(
    GEN_MODEL, torch_dtype=torch.float16, variant="fp16", use_safetensors=True,
    vae=AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16),   # VAE เดิมใน fp16 ได้รูปดำ
    image_encoder=CLIPVisionModelWithProjection.from_pretrained(
        "h94/IP-Adapter", subfolder="models/image_encoder", torch_dtype=torch.float16))
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, algorithm_type="dpmsolver++", use_karras_sigmas=True)
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter-plus_sdxl_vit-h.safetensors")
pipe.set_progress_bar_config(disable=True)
pipe.to("cuda")   # ~9GB อยู่บน T4 ทั้งตัว (เครื่องนี้ไม่มี SAM 3 มาแย่ง)
inpaint = StableDiffusionXLInpaintPipeline(**pipe.components)   # ใช้โมเดลชุดเดียวกัน ไม่กิน GPU เพิ่ม
inpaint.set_progress_bar_config(disable=True)
print(f"พร้อม: {GEN_MODEL} · GPU ว่างเหลือ {torch.cuda.mem_get_info()[0] / 2**30:.1f}GB")

NUM_WORDS = {1: "one", 2: "two", 3: "three", 4: "four", 5: "five", 6: "six", 7: "seven", 8: "eight", 9: "nine", 10: "ten"}


def tile(im, q=1):
    """ถ้าดอกไม้มี 1 หรือ 2 ดอก ห้ามทำตาราง 4 ดอกเด็ดขาด! ให้ส่งภาพเดี่ยวไปเพื่อคุมจำนวนดอกให้ตรงตามจริง"""
    if q <= 2:
        return im
    w, h = im.size
    small, out = im.resize((w // 2, h // 2)), Image.new("RGB", (w, h), "white")
    for x in (0, w // 2):
        for y in (0, h // 2):
            out.paste(small, (x, y))
    return out


def zone_images(n, total_stems=None, top=0.08, bottom=0.60):
    """สร้าง Mask แบบคลัสเตอร์ธรรมชาติ (Organic Clustered Ellipses) ซ้อนเหลื่อมกัน
    และปรับขนาดตามจำนวนดอกไม้จริงเพื่อไม่ให้งอกดอกเกิน"""
    ims = []
    if total_stems == 2 and n == 2:
        configs = [(0.43, 0.35, 0.18, 0.16), (0.57, 0.35, 0.18, 0.16)]
    else:
        cluster_configs = [
            [(0.42, 0.35, 0.28, 0.20), (0.58, 0.34, 0.28, 0.20)],
            [(0.40, 0.38, 0.25, 0.18), (0.60, 0.38, 0.25, 0.18), (0.50, 0.24, 0.26, 0.16)],
            [(0.38, 0.36, 0.22, 0.16), (0.62, 0.36, 0.22, 0.16), (0.50, 0.22, 0.24, 0.14), (0.50, 0.45, 0.22, 0.14)]
        ]
        configs = cluster_configs[min(max(0, n - 2), len(cluster_configs) - 1)]

    for i in range(n):
        m = Image.new("L", (GEN_W, GEN_H), 0)
        cx_rel, cy_rel, rx_rel, ry_rel = configs[i % len(configs)]
        cx, cy = int(GEN_W * cx_rel), int(GEN_H * cy_rel)
        rx, ry = int(GEN_W * rx_rel), int(GEN_H * ry_rel)
        ImageDraw.Draw(m).ellipse([cx - rx, cy - ry, cx + rx, cy + ry], fill=255)
        ims.append(m)
    return ims


def zone_masks(n, total_stems=None):
    t = IPAdapterMaskProcessor().preprocess(zone_images(n, total_stems=total_stems), height=GEN_H, width=GEN_W)   # [n,1,H,W]
    return [t.reshape(1, n, t.shape[2], t.shape[3])]


_gpu = threading.Lock()


def generate(body):
    """body จาก notebook วิเคราะห์: prompt, negative, lead, refs=[{png, plural, phrase, color, label, q}], total_stems, seed"""
    refs_data = body.get("refs") or []
    total_stems = body.get("total_stems") or sum(r.get("q", 1) for r in refs_data) or (1 if refs_data else 0)

    # ดึงรูปอ้างอิง: ถ้าดอกไม้ 1-2 ดอก ไม่ tile 2x2
    refs = []
    for r in refs_data[:4]:
        im = Image.open(io.BytesIO(base64.b64decode(r["png"]))).convert("RGB")
        q_val = r.get("q", 1)
        refs.append((tile(im, q_val), r))

    seed = int(body.get("seed") or time.time())

    # เสริม Negative Prompt ป้องกันการงอกดอกเกิน / แจกันซ้ำ / สีปนกัน
    extra_neg = ", two vases, multiple vases, double vase, duplicate vase, split vase, side by side, split image, split view, divided screen, half and half, two bouquets, multiple bouquets, hybrid flower, blended species, deformed petals, macro close-up, giant bloom, blurry blob"
    if total_stems == 1:
        extra_neg += ", multiple flowers, two flowers, three flowers, bouquet, bunch, cluster, crowded flowers, extra blooms, extra stems"
    elif total_stems == 2:
        extra_neg += ", three flowers, four flowers, five flowers, bouquet, bunch of flowers, crowded, extra blooms, extra stems, single flower"

    neg = (body.get("negative") or "") + extra_neg
    base_prompt = body["prompt"]

    # กำหนดค่า scale และ conditioning ตามจำนวนดอกไม้
    if total_stems == 1:
        scale = 0.22 if refs else 0.0
        ip_img = refs[0][0] if refs else Image.new("RGB", (224, 224), "white")
        xkw = None
    elif len(refs) >= 2:
        scale = [[REF_SCALE_ZONE] * len(refs)]
        ip_img = [[im for im, _ in refs]]
        xkw = {"ip_adapter_masks": zone_masks(len(refs), total_stems=total_stems)}
    else:
        scale = (0.22 if total_stems <= 2 else REF_SCALE) if refs else 0.0
        ip_img = refs[0][0] if refs else Image.new("RGB", (224, 224), "white")
        xkw = None

    with _gpu, torch.inference_mode():
        pipe.set_ip_adapter_scale(scale)
        im = pipe(prompt=base_prompt, negative_prompt=neg, ip_adapter_image=ip_img, cross_attention_kwargs=xkw,
                  num_inference_steps=GEN_STEPS, guidance_scale=6.5, width=GEN_W, height=GEN_H,
                  generator=torch.Generator("cpu").manual_seed(seed)).images[0]

        # Inpainting รอบสอง: ทำเฉพาะเมื่อมีหลายสายพันธุ์ เพื่อแต่งรายละเอียดและบล็อกสีปนกัน
        if len(refs) >= 2:
            pipe.set_ip_adapter_scale(0.24)
            z_imgs = zone_images(len(refs), total_stems=total_stems)
            for i, ((r, meta), z) in enumerate(zip(refs, z_imgs)):
                if not meta.get("phrase"):
                    continue
                col = meta.get("color", "")
                q_val = meta.get("q", 1)

                # บล็อกสีและสายพันธุ์อื่นในโซนนี้อย่างเด็ดขาด เพื่อป้องกันสีปนกัน
                other_colors = [m["color"] for _, m in refs if m.get("color") and m["color"].lower() != col.lower()]
                other_species = [m["plural"] for _, m in refs if m.get("plural") and m["plural"].lower() != meta.get("plural", "").lower()]

                color_neg = ", ".join(f"{c}, {c} petals, {c} {meta.get('plural', 'flowers')}" for c in other_colors)
                species_neg = ", ".join(other_species)

                q_phrase = "exactly one solitary" if q_val == 1 else f"exactly {NUM_WORDS.get(q_val, str(q_val))}"
                inpaint_prompt = f"beautiful {q_phrase} {meta['phrase']} with distinct {col} petals, crisp flower bloom, natural stems, boutique florist photo, sharp focus, 8k photorealistic"

                inpaint_neg = f"{neg}, vase, container, glass, pot, mug, wrapping paper, hands, two vases, side by side, {color_neg}, {species_neg}, hybrid flower, color bleeding, blended petals"
                if q_val <= 2:
                    inpaint_neg += ", crowded flowers, extra flowers, too many flowers, giant bloom, blurry blob"

                im = inpaint(prompt=inpaint_prompt,
                             negative_prompt=inpaint_neg,
                             image=im, mask_image=inpaint.mask_processor.blur(z, 48),
                             ip_adapter_image=r, strength=REFINE_STRENGTH, num_inference_steps=GEN_STEPS,
                             guidance_scale=6.5, width=GEN_W, height=GEN_H,
                             generator=torch.Generator("cpu").manual_seed(seed + 100 + i)).images[0]
    return im



## 2) ทดสอบ (ข้ามได้)

In [ ]:
# ทดสอบเจน 1 รูป (ไม่มีรูปอ้างอิง) — เช็กว่าโมเดลทำงาน
t0 = time.time()
test = generate({"prompt": "A hand-tied bouquet held in hands, stems wrapped in kraft paper with a ribbon, made of 7 pink tulips, "
                           "5 yellow sunflowers with big dark brown centers. Florist photo, plain light background, photorealistic",
                 "negative": "cartoon, drawing, blurry, deformed petals, text, watermark, hybrid flower", "seed": 0})
print(f"{time.time() - t0:.0f} วินาที")
test.resize((416, 512))

## 3) เปิดเซิร์ฟเวอร์ → เอาลิงก์ไปใส่ GEN_URL

In [ ]:
from fastapi import Body, FastAPI
from fastapi.responses import JSONResponse
import uvicorn

PORT = 8001
api = FastAPI()


@api.get("/")
def health():
    return {"ok": True, "model": GEN_MODEL}


@api.post("/generate")
def api_generate(body: dict = Body(...)):
    if not isinstance(body.get("prompt"), str) or not body["prompt"]:
        return JSONResponse({"error": "no_prompt"}, 400)
    t0 = time.time()
    im = generate(body)
    buf = io.BytesIO()
    im.save(buf, "JPEG", quality=90)
    print(f"[gen] {time.time() - t0:.0f} วินาที · {len(body.get('refs') or [])} รูปอ้างอิง · {body['prompt'][:90]}", flush=True)
    return {"image": "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()}


if globals().get("_server") is None:   # รันเซลล์ซ้ำ → ไม่เปิดพอร์ตซ้ำ
    _server = uvicorn.Server(uvicorn.Config(api, host="0.0.0.0", port=PORT, log_level="warning"))
    threading.Thread(target=_server.run, daemon=True).start()
    time.sleep(2)

if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q", "-O", "cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    os.chmod("cloudflared", 0o755)
if globals().get("_tunnel_url") is None or _tun.poll() is not None:   # tunnel ตาย → เปิดใหม่ ได้ลิงก์ใหม่
    _tunnel_url = None
    _tun = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"], stderr=subprocess.PIPE, text=True)
    t0 = time.time()
    for line in _tun.stderr:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            _tunnel_url = m.group(0)
            break
        if time.time() - t0 > 60:
            break
    threading.Thread(target=lambda: [None for _ in _tun.stderr], daemon=True).start()   # อ่านทิ้ง กัน pipe เต็ม
print("🎨 GEN_URL =", repr(globals().get("_tunnel_url")), "← คัดลอกไปใส่ใน notebook วิเคราะห์ ข้อ 2) ตั้งค่า")
print("ปล่อยแท็บนี้เปิดไว้ระหว่างใช้งาน · log การเจนจะขึ้นด้านล่าง")

## 4) (ใช้ตอนโค้ดบน GitHub ถูกแก้) อัปเดตโดยไม่โหลดโมเดลใหม่


In [ ]:
# อัปเดตโค้ดเจนรูปจาก GitHub โดยไม่โหลดโมเดลใหม่ (ลิงก์ GEN_URL เดิมใช้ต่อได้)
import json, urllib.request
_nb = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/039-Nippitch/flower/main/bloom_craft_gen_sdxl.ipynb"))
_src = "".join(_nb["cells"][3]["source"])
exec(_src[_src.index("GEN_STEPS"):_src.index("assert torch")])   # ค่าตั้งค่า
exec(_src[_src.index("def zone_images"):])                       # ฟังก์ชันเจนรูป
print("อัปเดตแล้ว · REF_SCALE_ZONE =", REF_SCALE_ZONE)